# Actividad calificable — Corte 3: Análisis temporal y visualización

**Objetivo:** analizar una serie temporal de ventas, agregarla por periodo mediante `resample`, calcular una media móvil, describir tendencia y estacionalidad, construir dos visualizaciones y documentar una mejora SQL / DAX.

> Este cuaderno está preparado para ser ejecutado de principio a fin y subido al repositorio de GitHub de la clase.


## 1. Importación de librerías y carga de datos

La serie contiene ventas diarias durante 2024 y 2025. Se usa una frecuencia diaria para después realizar la agregación mensual.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd()
archivo = BASE / "ventas_diarias.csv"

df = pd.read_csv(archivo, parse_dates=["fecha"])
df = df.sort_values("fecha")
df.head()


## 2. Preparación de la serie temporal

Se convierte `fecha` en índice temporal. Después, las ventas diarias se agregan a frecuencia mensual usando `resample("MS").sum()`.


In [ ]:
serie = df.set_index("fecha")["ventas"]
mensual = serie.resample("MS").sum().to_frame("ventas_mensuales")
mensual.head()


## 3. Media móvil

Se calcula una media móvil de **3 meses**. Esta técnica suaviza las fluctuaciones de corto plazo y permite observar con mayor claridad la tendencia.


In [ ]:
mensual["media_movil_3m"] = mensual["ventas_mensuales"].rolling(window=3).mean()
mensual.round(2)


## 4. Tendencia

Se compara el promedio de los primeros tres meses con el promedio de los últimos tres meses para resumir cuantitativamente el cambio de nivel.


In [ ]:
primeros_3 = mensual["ventas_mensuales"].iloc[:3].mean()
ultimos_3 = mensual["ventas_mensuales"].iloc[-3:].mean()
crecimiento = (ultimos_3 / primeros_3 - 1) * 100

print(f"Promedio primeros 3 meses: {primeros_3:,.2f}")
print(f"Promedio últimos 3 meses: {ultimos_3:,.2f}")
print(f"Crecimiento aproximado: {crecimiento:.1f}%")


**Interpretación:** el promedio de los últimos tres meses es aproximadamente **33.7%** superior al de los primeros tres meses. Esto sugiere una **tendencia general creciente**, que también debe confirmarse visualmente con la media móvil.


## 5. Visualización 1 — evolución y media móvil

El título comunica la conclusión principal. Para una serie temporal de líneas no es obligatorio iniciar el eje Y en cero, porque el objetivo es observar trayectoria y cambio a través del tiempo, no comparar longitudes de barras.


In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(mensual.index, mensual["ventas_mensuales"], marker="o", linewidth=1.5, label="Ventas mensuales")
plt.plot(mensual.index, mensual["media_movil_3m"], linewidth=2.6, label="Media móvil 3 meses")
plt.title("Las ventas muestran una tendencia creciente suavizada por la media móvil")
plt.xlabel("Fecha")
plt.ylabel("Ventas agregadas")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 6. Estacionalidad

Para evaluar un patrón recurrente se calcula el promedio de ventas para cada mes del año, combinando los dos años disponibles.


In [ ]:
mensual["mes"] = mensual.index.month
estacionalidad = mensual.groupby("mes")["ventas_mensuales"].mean().reindex(range(1,13))

nombres = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
tabla_estacional = pd.DataFrame({
    "mes": nombres,
    "ventas_promedio": estacionalidad.values
})
tabla_estacional


**Interpretación:** dentro de esta serie, **May** es el mes con mayor promedio de ventas y **Ene** el de menor promedio. La repetición de diferencias entre meses sugiere un componente estacional además de la tendencia.


## 7. Visualización 2 — comparación estacional

Como esta gráfica usa barras, el eje Y comienza en **cero** para que la comparación visual sea honesta y no amplifique artificialmente las diferencias.


In [ ]:
plt.figure(figsize=(10, 5.5))
plt.bar(nombres, estacionalidad.values)
plt.title("La comparación mensual permite identificar la estacionalidad de las ventas")
plt.xlabel("Mes del año")
plt.ylabel("Ventas mensuales promedio")
plt.ylim(bottom=0)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


## 8. Optimización SQL

### Consulta original

```sql
SELECT
    YEAR(fecha) AS anio,
    MONTH(fecha) AS mes,
    SUM(ventas) AS ventas_mensuales
FROM ventas
WHERE YEAR(fecha) = 2025
GROUP BY YEAR(fecha), MONTH(fecha)
ORDER BY anio, mes;
```

### Consulta optimizada

```sql
SELECT
    DATE_TRUNC('month', fecha) AS mes,
    SUM(ventas) AS ventas_mensuales
FROM ventas
WHERE fecha >= DATE '2025-01-01'
  AND fecha <  DATE '2026-01-01'
GROUP BY DATE_TRUNC('month', fecha)
ORDER BY mes;

CREATE INDEX idx_ventas_fecha ON ventas(fecha);
```

**Explicación de la mejora:** aplicar `YEAR(fecha)` dentro del `WHERE` obliga al motor a transformar la columna antes de evaluar el filtro y puede impedir el uso eficiente de un índice. Un filtro por rango (`>=` y `<`) es más amigable con índices B-tree sobre la fecha y reduce el conjunto de filas que deben procesarse. `DATE_TRUNC` se usa únicamente para agrupar el resultado final por mes.


## 9. Medidas DAX

```DAX
Ventas Totales =
SUM(Ventas[ventas])
```

```DAX
Promedio Móvil 3M =
VAR FechaMax = MAX(Calendario[Fecha])
RETURN
AVERAGEX(
    DATESINPERIOD(
        Calendario[Fecha],
        FechaMax,
        -3,
        MONTH
    ),
    CALCULATE([Ventas Totales])
)
```

**Mejora de la métrica:** `Ventas Totales` centraliza la definición del indicador base y permite reutilizarlo. `Promedio Móvil 3M` reduce el ruido de corto plazo y facilita comparar la dirección del negocio entre periodos. Una tabla calendario dedicada hace más consistente la inteligencia de tiempo.


## 10. Findings

The monthly sales series shows an overall upward trend across the two-year period.  
The three-month moving average makes this trend easier to interpret because it reduces short-term fluctuations.  
The first chart compares monthly sales with the moving average and shows that the smoothed series follows the same general direction with less noise.  
The second chart compares average sales by calendar month and reveals a recurring seasonal pattern, with May showing the highest average level in this dataset.  
Because the bar chart starts at zero, the magnitude of the differences between months is represented honestly.  
The optimized SQL filter is more index-friendly than applying a year function directly to the date column, while the DAX moving-average measure provides a clearer business indicator for trend monitoring.


## Bibliografía

Referencias técnicas consultadas y utilizadas como soporte metodológico:

- McKinney, W. (2022). Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter (3rd ed.). O'Reilly Media.
- pandas development team. (2026). Resampling. pandas 3.1.1 documentation. https://pandas.pydata.org/docs/reference/resampling.html
- pandas development team. (2026). pandas.DataFrame.rolling. pandas documentation. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html
- Matplotlib development team. (2026). Matplotlib documentation. https://matplotlib.org/stable/
- PostgreSQL Global Development Group. (2026). Date/Time Functions and Operators: date_trunc. PostgreSQL 16 Documentation. https://www.postgresql.org/docs/16/functions-datetime.html
- PostgreSQL Global Development Group. (2026). Indexes. PostgreSQL Documentation. https://www.postgresql.org/docs/current/indexes.html
- Microsoft. (2026). Time intelligence functions (DAX). Microsoft Learn. https://learn.microsoft.com/dax/time-intelligence-functions-dax
- Microsoft. (2026). DATESINPERIOD function (DAX). Microsoft Learn. https://learn.microsoft.com/dax/datesinperiod-function-dax
- Microsoft. (2026). AVERAGEX function (DAX). Microsoft Learn. https://learn.microsoft.com/dax/averagex-function-dax
- GitHub Docs. (2026). Git basics and working with repositories. https://docs.github.com/
